# Training Lora From Corpus Version 2 (Big)
This adds stop tokens to the training

# Load model with Unsloth patching

In [1]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-8B",
    max_seq_length=1024,
    load_in_4bit=True,
)

print("Loaded model in 4-bit ✅")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 11-05 11:34:56 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.9: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    NVIDIA RTX A5000. Num GPUs = 1. Max memory: 23.673 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+e98c69b.d20251102. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loaded model in 4-bit ✅


# Apply LoRa adapter

In [2]:
peft_model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # Keep: balances capacity and efficiency
    lora_alpha=32,  # Lowered: reduces overfitting risk (scale = 1)
    lora_dropout=0.1,  # Slightly higher: better regularization for noisy esoteric texts
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"  # Attention: core for reasoning
        #"up_proj", "down_proj", "gate_proj"  # MLP: adds expressivity for complex patterns
    ],
    bias="none",  # Keep: minimizes params
    use_gradient_checkpointing=True,  # Keep: VRAM saver
    modules_to_save=None,  # Default: avoids retraining embeddings
    use_rslora=True  # NEW: rank-stabilized LoRA, improves stability for higher r
)

tokenizer.bos_token = None
tokenizer.eos_token = "</s>"  # Qwen3’s typical EOS token
tokenizer.pad_token = tokenizer.eos_token  # Common practice
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
peft_model.config.bos_token_id = tokenizer.bos_token_id
peft_model.config.eos_token_id = tokenizer.eos_token_id
peft_model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded peft model ✅")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.9 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Loaded peft model ✅


# Load the dataset from corpus

In [3]:
import os 

#CORPUS_DIR = "/storage/corpus/wtk_archive_with_stops"
CORPUS_DIR = "/storage/corpus/corpus_health_tiny"

BLOCK_SIZE = 1024  # max tokens per chunk

tok = tokenizer 

# Ensure EOS/PAD exist and are consistent
added = False
if tok.eos_token is None:
    tok.add_special_tokens({"eos_token": "</s>"})
    added = True
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    added = True
if added:
    model.resize_token_embeddings(len(tok))

# -----------------------
# 2) Load raw text files (no EOS strings here)
# -----------------------
def load_txt_corpus(directory):
    texts = []
    for filename in os.listdir(directory):
        if not filename.endswith(".txt"):
            continue
        path = os.path.join(directory, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read().strip()
            if txt:
                texts.append(txt)
    return texts

raw_texts = load_txt_corpus(CORPUS_DIR)
print(f"Loaded {len(raw_texts)} files from {CORPUS_DIR} ✅")

Loaded 7387 files from /storage/corpus/corpus_health_tiny ✅


# Tokenize with EOS appended (token id, not string)
We’ll build one long stream of ids and then pack into BLOCK_SIZE chunks.

In [4]:
from datasets import Dataset

def tokenize_append_eos(texts):
    # batch tokenize; append eos token string so tokenizer emits eos_token_id
    # Alternatively: add eos id manually after each example (equivalent).
    enc = tok(texts, add_special_tokens=False)
    input_ids = []
    for ids in enc["input_ids"]:
        input_ids.extend(ids)
        if tok.eos_token_id is not None:
            input_ids.append(tok.eos_token_id)
    return input_ids

flat_ids = tokenize_append_eos(raw_texts)

# -----------------------
# 4) Pack into fixed-length blocks (no cross-doc bleed because we injected EOS)
# -----------------------
def pack_ids_to_blocks(ids, block_size):
    blocks = []
    for i in range(0, len(ids) - block_size + 1, block_size):
        chunk = ids[i : i + block_size]
        blocks.append({"input_ids": chunk, "attention_mask": [1] * len(chunk)})
    return Dataset.from_list(blocks)

train_dataset = pack_ids_to_blocks(flat_ids, BLOCK_SIZE)
print(f"Prepared {len(train_dataset)} packed training chunks of {BLOCK_SIZE} tokens ✅")


Prepared 22582 packed training chunks of 1024 tokens ✅


# Init Trainer Params

In [6]:
import os
from transformers import TrainingArguments
from trl import SFTTrainer

# 1) Absolute, writable, persistent output dir
OUTPUT_DIR = "/workspace/wtk-qwen3-8b-health-lora-v2"

# 2) Build explicit TrainingArguments (NO dict here)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    resume_from_checkpoint=True,  # Fresh start unless resuming
    num_train_epochs=2,  # Increased: 600 MB (~300K–600K tokens) needs 1–2 passes
    per_device_train_batch_size=2,  # Lowered: fits A4000 with expanded modules
    gradient_accumulation_steps=4,  # Balances batch size (~8 effective)
    lr_scheduler_type="cosine",  # Keep: stable for esoteric data
    learning_rate=2e-5,  # Slightly higher: suits smaller dataset, lora_alpha=32
    warmup_ratio=0.05,  # Adjusted: gradual ramp for stability
    weight_decay=0.01,  # Keep: prevents overfitting
    fp16=False,  # Keep: A4000 supports BF16
    bf16=True,  # Keep: efficient on Ampere
    logging_steps=10,  # Keep: frequent monitoring
    save_steps=250,  # Keep: regular checkpoints
    report_to="none",
    remove_unused_columns=False,
    metric_for_best_model="loss",
    greater_is_better=False,
    max_grad_norm=0.5,  # Lowered: stabilizes expanded modules
    dataloader_num_workers=0  # Keep: stable on mounted storage
)

# 3) Build the trainer
trainer = SFTTrainer(
    model=peft_model,
    tokenizer=tok,
    train_dataset=train_dataset,
    max_seq_length=BLOCK_SIZE,
    args=training_args,
)

print(f"Created SFTTrainer ✅")


Created SFTTrainer ✅


# Train using SFTTrainer (new)

In [ ]:
# 4) Start training (this might take a while)
trainer.train()
print("Training complete ✅")

# 5) Save trained model to storage
trainer.model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128247, 'bos_token_id': None, 'pad_token_id': 128247}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 148,673 | Num Epochs = 2 | Total steps = 37,170
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.432700
20,2.449200
30,2.471800
40,2.352900
50,2.427400
60,2.522300
70,2.416100
80,2.493500
90,2.510900
100,2.326000


# Resume training using SFTTrainer (only use if resuming)

In [ ]:
CHECKPOINT_DIR = OUTPUT_DIR + "/" + "checkpoint-2000"

#4) Start training (this might take a while)
print(f"Resuming from checkpoint: {CHECKPOINT_DIR}")
trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)
print("Training complete ✅")

# 5) Save trained model to storage
trainer.model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128247, 'bos_token_id': None, 'pad_token_id': 128247}.


Resuming from checkpoint: /workspace/wtk-qwen3-8b-health-lora-v2/checkpoint-2000


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 22,582 | Num Epochs = 2 | Total steps = 5,646
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 30,670,848 of 8,221,406,208 (0.37% trained)
	save_steps: 250 (from args) != 1000 (from trainer_state.json)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


# Push LoRa to Huggingface

In [1]:
from huggingface_hub import HfApi, upload_folder

repo_id = "peers-ai/wtk-qwen3-beta-slim-lora-v3"
folder = "/storage/models/wtk-qwen3-beta-slim-lora-v3"  # contains adapter_config.json & adapter_model.bin

api = HfApi()
# create the repo if it doesn't exist
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)

# upload all files in the folder
upload_folder(
    repo_id=repo_id,
    folder_path=folder,
    repo_type="model",
)
print(f"✅ Uploaded to https://huggingface.co/{repo_id}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Uploaded to https://huggingface.co/peers-ai/wtk-qwen3-beta-slim-lora-v3
